<a href="https://colab.research.google.com/github/babessell1/GWC_Test/blob/main/2D_Animations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ipympl
from google.colab import output
output.enable_custom_widget_manager()
# restart the runtime after running this block!

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.8 MB/s eta 0:00:00


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib ipympl

In [ ]:
# variable cheat-sheet
# ==========================
# y(x,t)  - instantaneous wave value (what we visualize)
# A       - amplitude (peak value)
# x, z    - spatial coordinates (1D or 2D)
# Δx      - displacement / source position shift
# λ       - wavelength (distance per cycle)
# k       - wave number = 2π / λ (radians per unit length)
# f       - frequency in cycles/second (Hz)
# ω       - angular frequency = 2π f (radians/second)
# v       - wave speed (length / second). For simple non-dispersive waves: ω = k * v
# φ       - phase offset (radians). If provided in degrees, convert with np.deg2rad()
# r       - distance from a point source to the field point (sqrt(dx^2 + dy^2) in 2D)
# t       - time (seconds)
# i       - the imaginary unit, i² = −1. (it lets us represent a 2D arrow.)

#Phasor trick
Imagine every sine wave as a little spinning arrow (like a clock hand). The wave value is just the arrow’s vertical shadow. Instead of recomputing lots of sines every frame, we pre‑draw the arrows and spin them — cheap and fast.

The clock‑hand picture (no heavy math)

Take an arrow of length A. Put its tail at the origin.  
Point it at angle θ. The arrow’s vertical shadow (up/down) = A · sin(θ).  
Now make that arrow rotate smoothly — its vertical shadow wiggles in time → it makes a sine wave.

So: sine = vertical shadow of a spinning arrow.

In [3]:
# Euler’s formula:
# e^{iθ} = cos θ + i·sin θ  ⇒  sin θ = Im(e^{iθ}) ... (Im means the imaginary part!)

# Start with the usual wave:
# y(x,t) = A · sin(kx − ωt + φ)

# Rewrite it using eulers formula:
# y(x,t) = Im{ A · e^{i(kx − ωt + φ)} }

# Factor time out:
# y(x,t) = Im{ [A · e^{i(kx + φ)}] · e^{−iωt} }

# Define the spatial arrow (phasor):
# C(x) = A · e^{i(kx + φ)}
#  — this is a fixed arrow at each x.

# Now y(x,t) = Im{ C(x) · e^{−iωt} }
#  — rotate every arrow by the same amount e^{−iωt} and take its vertical shadow

In [ ]:
# ==============================================================================
# Part A — Small 1-D sanity check: traditional sin() vs phasor formulation
# ==============================================================================

def sine_wave_traditional(x, amplitude=1.0, phase_deg=0.0, displacement=0.0, wavelength=1.0):
    """
    The usual real-valued sine definition for 1-D demonstrations:
      y(x) = A * sin( k*(x - displacement) + phi )
    """
    phi = np.deg2rad(float(phase_deg))
    k = 2.0 * np.pi / float(wavelength)
    return amplitude * np.sin(k * (x - displacement) + phi)


def sine_wave_phasor(x, amplitude=1.0, phase_deg=0.0, displacement=0.0, wavelength=1.0, time=0.0, wave_speed=1.0):
    """
    The phasor-based version: build the complex phasor C(x) = A * exp(i*(k*(x-dx) + phi))
    and then return the instantaneous real wave y(x,t) = Im( C(x) * exp(-i * omega * t) ).
    This shows the algebraic equivalence to the traditional sine.
    """
    phi = np.deg2rad(float(phase_deg))
    k = 2.0 * np.pi / float(wavelength)
    omega = 2.0 * np.pi * wave_speed / float(wavelength)
    C = amplitude * np.exp(1j * (k * (x - displacement) + phi))   # complex spatial phasor
    y = np.imag(C * np.exp(-1j * omega * time))                   # rotate in time, then take imaginary part
    return y


# double check!
# are the functions equivalent for t=0 (and at any t, if we include time)?
xs = np.linspace(0, 1, 50)
y_trad = sine_wave_traditional(xs, amplitude=1.23, phase_deg=30, displacement=0.1, wavelength=0.3)
y_phas = sine_wave_phasor(xs, amplitude=1.23, phase_deg=30, displacement=0.1, wavelength=0.3, time=0.0)
assert np.allclose(y_trad, y_phas, atol=1e-12), "Traditional and phasor versions must match (t=0)."